# Infer CNV

.. warning::
    We consider this method still experimental, but decided to already put it on GitHub because it might be useful. 
    Treat the results with care. In particular, no validation with ground-truth data has been performed. 

In [ ]:
import scanpy as sc
import infercnvpy as cnv
import matplotlib.pyplot as plt
import warnings

warnings.simplefilter("ignore")

# sc.settings.set_figure_params(figsize=(5, 5))

sc.logging.print_header()

## Loading the example dataset

.. note::

    **Preprocessing data**
    
    Low-quality cells should already be filtered out and the input data 
    must be normalized and log-transformed. For more information, see
    :ref:`input-data`. 
    
    Also, the genomic positions need to be stored in `adata.var`. The 
    columns `chromosome`, `start`, and `end` hold the chromosome and 
    the start and end positions on that chromosome for each gene, 
    respectively. 
    
    Infercnvpy provides the :func:`infercnvpy.io.genomic_position_from_biomart` and 
    :func:`infercnvpy.io.genomic_position_from_gtf` functions
    to get these information online or from a GTF file and store them in `adata.var`. 
    

In [ ]:
input_file = '../../results/merged/chosen_branch/adata.h5ad'
windows_size = 100

In [ ]:
adata = sc.read_h5ad(input_file)
adata.X = adata.layers['log1p_norm_of_counts']

In [ ]:
if 'ensembl_id' in adata.var_keys():
	adata_gene_id = 'ensembl_id'
	biomart_gene_id = 'ensembl_gene_id'
elif all(adata.var_names.str.startswith('ENSG')):
	adata_gene_id = None
	biomart_gene_id = 'ensembl_gene_id'
else:
	adata_gene_id = None
	biomart_gene_id = 'hgnc_symbol'

cnv.io.genomic_position_from_biomart(adata, adata_gene_id = adata_gene_id, biomart_gene_id = biomart_gene_id, inplace = True)

## Running infercnv

Let's now run :func:`infercnvpy.tl.infercnv`. Essentially, this method sorts genes
by chromosome and genomic position and compares the average gene expression over genomic
region to a reference. The original inferCNV method uses a window size of 100, 
but larger window sizes can make sense, depending on the number of 
genes in your dataset. 

:func:`~infercnvpy.tl.infercnv` adds a `cell x genomic_region` matrix to 
`adata.obsm["X_cnv"]`. 

For more information about the method check out :ref:`infercnv-method`. 

.. note::

    **Choosing reference cells**
    
    The most common use-case is to compare tumor against normal cells. If you have 
    prior information about which cells are normal (e.g. from cell-type annotations
    based on transcriptomics data), it is recommended to provide this information 
    to :func:`~infercnvpy.tl.infercnv`. 
    
    The more different cell-types you 
    can provide, the better. Some cell-types physiologicallly over-express
    certain genomic regions (e.g. plasma cells highly express Immunoglobulin genes
    which are genomically adjacent). If you provide multiple cell-types, 
    only regions are considered being subject to CNV that are different from *all*
    provided cell-types. 
    
    If you don't provide any reference, the mean of all cells is used instead,
    which may work well on datasets that contain enough tumor and normal cells. 

In [ ]:
cnv.tl.infercnv(
    adata,
    reference_key=None,
    reference_cat=None,
    window_size=windows_size,
)

Now, we can plot smoothed gene expression by cell-type and chromosome. 
We can observe that the Epithelial cell cluster, which consists largely of tumor cells, appears
to be subject to copy number variation. 

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby="sample")

## Clustering by CNV profiles and identifying tumor cells

To cluster and annotate cells, `infercnvpy` mirrors the `scanpy` workflow. 
The following functions work exactely as their `scanpy` counterpart, except that 
they use the CNV profile matrix as input. Using these functions, we can perform
graph-based clustering and generate a UMAP plot based on the CNV profiles. 
Based on these clusters, we can annotate tumor and normal cells. 

.. module:: infercnvpy
   :noindex:

.. autosummary::
       
   infercnvpy.tl.pca
   infercnvpy.pp.neighbors
   infercnvpy.tl.leiden
   infercnvpy.tl.umap
   infercnvpy.pl.umap

In [ ]:
cnv.tl.pca(adata)
cnv.pp.neighbors(adata)
cnv.tl.leiden(adata)

After running leiden clustering, we can plot the chromosome heatmap 
by CNV clusters. We can observe that, as opposted to the clusters 
at the bottom, the clusters at the top have essentially no differentially expressed genomic regions. 
The differentially expressed regions are likely due to copy number variation and the respective 
clusters likely represent tumor cells. 

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby="cnv_leiden", dendrogram=True)

### UMAP plot of CNV profiles

We can visualize the same clusters as a UMAP plot. Additionally, 
:func:`infercnvpy.tl.cnv_score` computes a summary score that quantifies the amount of copy
number variation per cluster. It is simply defined as the
mean of the absolute values of the CNV matrix for each cluster. 

In [ ]:
cnv.tl.umap(adata)
cnv.tl.cnv_score(adata)

The UMAP plot consists of a large blob of "normal" cells and several smaller clusters
with distinct CNV profiles. Except for cluster "12", which consists of ciliated cells, 
the isolated clusters are all epithelial cells. These are likely tumor cells and each 
cluster represents an individual sub-clone.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(11, 11))
ax4.axis("off")
cnv.pl.umap(
    adata,
    color="cnv_leiden",
    legend_loc="on data",
    legend_fontoutline=2,
    ax=ax1,
    show=False,
)
cnv.pl.umap(adata, color="cnv_score", ax=ax2, show=False)
cnv.pl.umap(adata, color="phase", ax=ax3)

In [ ]:
import numpy as np

# Custom plotting function
# adapted from https://github.com/scverse/scanpy/issues/2333#issuecomment-1563790561
def split_dim_plot(adata, split_by, embedding="umap", color=None, ncol=5, nrow=None, **kwargs):
    size = 120000 / len(adata.obs_names) * len(adata.obs[split_by].unique()) / 1.2  # Keep point size constant
    categories = adata.obs[split_by].cat.categories
    if nrow is None:
        nrow = int(np.ceil(len(categories) / ncol))
    
    # Update global font sizes
    plt.rcParams.update({
        "font.size": 14,  # General font size
        "axes.titlesize": 18,  # Subplot title size
        "legend.fontsize": 14,  # Legend font size
        "xtick.labelsize": 14,  # X-axis labels
        "ytick.labelsize": 14   # Y-axis labels
    })
    
    fig, axs = plt.subplots(nrow, ncol, figsize=(5*ncol, 5*nrow))
    axs = axs.flatten()

    # Plot individual subplots
    for i, cat in enumerate(categories):
        ax = axs[i]
        
        # Plot background (all cells in light gray)
        sc.pl.embedding(adata, basis=embedding, color=None, size=size, show=False, ax=ax, na_color="#EEEEEE")
        
        # Plot current category with color
        sc.pl.embedding(
            adata[adata.obs[split_by] == cat], 
            basis=embedding, 
            color=color, 
            ax=ax, 
            show=False, 
            title=cat, 
            size=size, 
            legend_loc=None,  # Remove individual legends
            **kwargs
        )
    
        ax.set_title(cat, fontsize=18)  # Set larger font for subplot titles

    # Hide unused subplots if any
    for j in range(i + 1, len(axs)):
        axs[j].axis("off")

    # Create common legend
    if color and color in adata.obs:
        unique_categories = adata.obs[color].cat.categories
        colors = adata.uns.get(color + "_colors", plt.cm.tab10(range(len(unique_categories))))  # Get colors
        
        legend_patches = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=12, label=cat)
                          for cat, c in zip(unique_categories, colors)]

        fig.legend(handles=legend_patches, loc="lower center", ncol=min(len(unique_categories), 5), fontsize=16)  # Bigger legend
        unique_categories = adata.obs[color].cat.categories
        colors = adata.uns.get(color + "_colors", plt.cm.tab10(range(len(unique_categories))))  # Get colors
        
        legend_patches = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=12, label=cat)
                          for cat, c in zip(unique_categories, colors)]

        fig.legend(handles=legend_patches, loc="lower center", ncol=min(len(unique_categories), 5), fontsize=16)  # Bigger legend

    # Adjust layout to make space for legend
    plt.tight_layout(rect=[0, 0.05, 1, 1])

    # plt.show()

In [ ]:
adata

In [ ]:
split_dim_plot(adata, 'sample', "X_cnv_umap", color='phase', ncol=5)

In [ ]:
split_dim_plot(adata, 'sample', "X_cnv_pca", color='phase', ncol=5)

We can also visualize the CNV score and clusters on the transcriptomics-based UMAP plot. 
Again, we can see that there are subclusters of epithelial cells that belong
to a distinct CNV cluster, and that these clusters tend to have the 
highest CNV score. 

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 11), gridspec_kw={"wspace": 0.5})
ax4.axis("off")
sc.pl.umap(adata, color="cnv_leiden", ax=ax1, show=False)
sc.pl.umap(adata, color="cnv_score", ax=ax2, show=False)
sc.pl.umap(adata, color="phase", ax=ax3)

### Classifying tumor cells

Based on these observations, we can now assign cell to either "tumor" or "normal". 
To this end, we add a new column `cnv_status` to `adata.obs`. 

In [ ]:
adata.obs["cnv_status"] = "normal"
adata.obs.loc[adata.obs["cnv_leiden"].isin(["10", "16", "13", "8", "12", "17", "1", "14", "11"]), "cnv_status"] = (
    "tumor"
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"wspace": 0.5})
cnv.pl.umap(adata, color="cnv_status", ax=ax1, show=False)
sc.pl.umap(adata, color="cnv_status", ax=ax2)

Now, we can plot the CNV heatmap for tumor and normal cells separately: 

In [ ]:
adata.obs["cnv_leiden"].isin(["8", "10", "11", "12", "14", "16", "17"])

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs["cnv_leiden"].isin(["6", "8", "10", "11", "12", "14", "16", "17"])])

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs["cnv_status"] == "tumor", :])

In [ ]:
cnv.pl.chromosome_heatmap(adata[adata.obs["cnv_status"] == "normal", :])

In [ ]:
# save adata
output_file = input_file.replace('.h5ad', '_updated.h5ad')
adata.write_h5ad(input_file)

In [ ]:
# # copykat
# adata.X = adata.layers['counts']
# cnv.tl.copykat(adata, key_added = 'copykat', gene_ids = 'S')